# Proven 分析记录

领域包：金融 · 宏观与微观经济　会话：`fc841d6385`

在项目根目录打开本 Notebook，运行下面的准备单元会重新加载数据，之后每个单元都可以复现报告中的数字。

In [ ]:
# 准备：导入与加载数据
import pandas as pd, numpy as np, duckdb
import plotly.express as px, plotly.graph_objects as go
from proven.kit.core import load_table, sql, tables, profile
from proven.kit.finance import *
_name = load_table('data/finance/capm_monthly.csv')
print(_name)
_name = load_table('data/finance/us_macro_quarterly.csv')
print(_name)
_name = load_table('data/finance/us_treasury_10y.csv')
print(_name)
_name = load_table('data/finance/world_macro.csv')
print(_name)

## 问题：2000 年以来中国、美国、日本、德国的 GDP 增速和通胀率有什么差异？

**计划**

1. 查口径与文档：用 lookup_semantic 查询「人均 GDP」「通胀率」的定义口径，用 search_knowledge 检索「宏观经济指标与关系」「数据说明与常见陷阱」中关于名义/实际、区域汇总行、缺失值处理的说明
2. 提取与检查数据：从 world_macro 筛选 CHN/USA/JPN/DEU 且 year>=2000、is_aggregate=False 的行，检查四国 gdp_usd、inflation_cpi_pct、inflation_deflator_pct、population 的缺失情况和有效年份范围，确定分析终点年份
3. 名义 GDP 增速：按国家计算 2000 年至终点年份的现价美元 GDP 年增长率（逐年）、期间几何平均年增速（CAGR）与累计增长倍数，横向比较四国
4. 实际 GDP 增速：用 inflation_deflator_pct 构造以 2000 年为基期的累积平减指数，折算各国实际 GDP 序列，计算实际 GDP 年均增速、人均实际 GDP 年均增速，并与名义增速对比（名义-实际的差值近似平减指数通胀贡献）
5. 通胀水平与波动：对四国计算 2000 年以来 CPI 通胀率与 GDP 平减指数通胀的均值、中位数、标准差、最大值与最小值，识别通缩年份（如日本的负通胀）
6. 分阶段对比与作图：把样本期分为 2000-2009、2010-2019、2020-终点三段，比较四国各阶段实际 GDP 增速与通胀均值的变化（如中国增速下台阶、日本长期低通胀），并用 plotly 画出四国实际 GDP 指数与通胀率的时间序列图

In [ ]:
# [C1]
codes = ['CHN','USA','JPN','DEU']
wm = world_macro[world_macro.code.isin(codes) & (world_macro.is_aggregate==False)]
print(wm.groupby('code').agg(n=('year','size'), yr_min=('year','min'), yr_max=('year','max')))
# 末年各列是否有值
sub = wm[wm.year>=2000]
print(sub.groupby('code').agg(
    n=('year','size'),
    gdp_miss=('gdp_usd', lambda s: s.isna().sum()),
    cpi_miss=('inflation_cpi_pct', lambda s: s.isna().sum()),
    def_miss=('inflation_deflator_pct', lambda s: s.isna().sum()),
    pop_miss=('population', lambda s: s.isna().sum()),
    yr_max=('year','max')))
print(sub[sub.year>=2020][['country','year','gdp_usd','inflation_cpi_pct','inflation_deflator_pct','population']].sort_values(['country','year']).to_string())

       n  yr_min  yr_max
code                    
CHN   66    1960    2025
DEU   66    1960    2025
JPN   66    1960    2025
USA   66    1960    2025
       n  gdp_miss  cpi_miss  def_miss  pop_miss  yr_max
code                                                    
CHN   26         2         1         2         0    2025
DEU   26         2         1         2         0    2025
JPN   26         2         1         2         0    2025
USA   26         2         1         2         0    2025
             country  year       gdp_usd  inflation_cpi_pct  inflation_deflator_pct    population
2700           China  2020  1.468774e+13           2.419422                0.492494  1.411100e+09
2701           China  2021  1.782046e+13           0.981015                4.552327  1.412360e+09
2702           China  2022  1.788178e+13           1.973576                1.823677  1.412175e+09
2703           China  2023  1.779478e+13           0.234837               -0.582394  1.410710e+09
2704           Chi

In [ ]:
# [C2]
d = wm[(wm.year>=2000)&(wm.year<=2023)].copy()
d = d.sort_values(['code','year'])
rows=[]
for c in codes:
    s = d[d.code==c].set_index('year')
    nom = s.gdp_usd
    deff = s.inflation_deflator_pct
    pop = s.population
    # 以 2000 年平减指数为 100 的累积价格指数
    P = (1+deff/100).cumprod()
    real = nom / P          # 按 2000 年价格衡量的实际 GDP（以 2000 年美元计）
    realpc = real/pop
    n = len(nom)
    yrs = s.index.max()-s.index.min()
    nom_g = growth_rate(nom, 1)         # 逐年名义增速 %
    real_g = growth_rate(real, 1)       # 逐年实际增速 %
    rows.append(dict(code=c,
        years=f"{s.index.min()}-{s.index.max()}", n=n,
        nom_cagr=((nom.iloc[-1]/nom.iloc[0])**(1/yrs)-1)*100,
        real_cagr=((real.iloc[-1]/real.iloc[0])**(1/yrs)-1)*100,
        realpc_cagr=((realpc.iloc[-1]/realpc.iloc[0])**(1/yrs)-1)*100,
        nom_cum=(nom.iloc[-1]/nom.iloc[0]-1)*100,
        real_cum=(real.iloc[-1]/real.iloc[0]-1)*100,
        real_mult=real.iloc[-1]/real.iloc[0],
        defl_cum=(P.iloc[-1]-1)*100,
        nom_minus_real=(((nom.iloc[-1]/nom.iloc[0])**(1/yrs)-1)-((real.iloc[-1]/real.iloc[0])**(1/yrs)-1))*100,
        real_g_std=real_g.std(ddof=1)))
res = pd.DataFrame(rows)
pd.set_option('display.width',200)
print(res.round(2).to_string(index=False))

code     years  n  nom_cagr  real_cagr  realpc_cagr  nom_cum  real_cum  real_mult  defl_cum  nom_minus_real  real_g_std
 CHN 2000-2023 24     12.39       8.99         8.47  1369.03    624.59       7.25    106.92            3.40        6.00
 USA 2000-2023 24      4.36       2.03         1.25   166.91     58.75       1.59     71.95            2.33        1.77
 JPN 2000-2023 24     -0.71      -0.52        -0.44   -15.20    -11.20       0.89     -5.78           -0.20        8.56
 DEU 2000-2023 24      3.66       1.75         1.69   128.75     48.99       1.49     53.01            1.91        7.95


## 问题：2000 年以来中国、美国、日本、德国的 GDP 增速和通胀率有什么差异？

**计划**

1. 核对数据覆盖：查询 world_macro 中 CHN/USA/JPN/DEU 四个非汇总经济体的年份范围，统计 2000 年以来 gdp_usd、population、inflation_cpi_pct、inflation_deflator_pct 的缺失情况，确定实际可用样本期。
2. 名义GDP增速对比：对四国的 gdp_usd 计算逐年同比增速（列 2001-2024 序列）与样本期 CAGR（cagr 函数），并报告均值、标准差与极值，用代码打印全部数字。
3. 人均GDP增速：用 gdp_usd/population 构造人均 GDP（现价美元），计算四国逐年增速与样本期 CAGR，剔除人口规模差异后再比较增速。
4. 通胀率对比：计算四国 2000 年以来 CPI 通胀率均值、标准差、最高/最低年份，并用 GDP 平减指数通胀做稳健性对照，另计算四国通胀均值之差。
5. 分阶段演变：把样本期切成 2000-2007、2008-2019、2020-2024 三段，分段计算四国的名义 GDP 增速与 CPI 通胀均值，比较排序是否随阶段改变。
6. 可视化与口径说明：用 plotly 画四国名义 GDP 增速与 CPI 通胀的时间序列折线图（双图或分面），标注口径为现价美元 GDP 与 CPI 同比。

## 问题：改成2015年以来

**计划**



## 问题：2000 年以来中国、美国的 GDP 增速和通胀率有什么差异？

**计划**

1. 核对两国数据可用性：在 world_macro 中筛出 code 为 CHN、USA 且 is_aggregate=False 的行，查看 2000-2025 年间 gdp_usd、inflation_cpi_pct、inflation_deflator_pct、population 的非缺失行数与缺失分布，确认可比年份区间与末年份是否完整。
2. 构造实际GDP与增速：用 inflation_deflator_pct 逐年累乘构造 GDP 平减指数（基期设在 2000 年），将名义 gdp_usd 折算为不变价 GDP，再计算两国 2000 年以来的年度实际增速、名义增速，以及全期几何平均增速（CAGR）和增速标准差。
3. 人均增速对比：用不变价 GDP 除以 population 得到人均实际 GDP，计算两国 2000 年以来人均实际增速的均值与累计涨幅，并拆出人口增速贡献（总量增速 - 人均增速 ≈ 人口增速）。
4. 通胀水平与波动：分别用 CPI 通胀率与 GDP 平减指数通胀率，计算两国 2000 年以来的均值、中位数、标准差、最高与最低年份，比较水平与波动性差异。
5. 分期段比较：把样本切成 2000-2007、2008-2019、2020-2025 三段，分别计算两国实际 GDP 增速均值与 CPI 通胀均值，观察 2020 年后美国通胀冲高与中国低通胀/通缩的阶段性分化。
6. 增速与通胀关联：对两国分别计算 2000 年以来实际 GDP 增速与 CPI 通胀率的相关系数（并给出样本量），作为增长—通胀关系的描述性证据，明确写为相关而非因果。

In [ ]:
# [C3]
import pandas as pd
mask = (wm['code'].isin(['CHN','USA'])) & (wm['is_aggregate']==False) & (wm['year']>=2000)
w2 = wm[mask].copy()
print(w2['code'].value_counts())
print(w2.groupby('code')['year'].agg(['min','max','count']))
print(w2.isna().sum())
print(w2[w2['code']=='CHN'].sort_values('year').to_string())

code
CHN    26
USA    26
Name: count, dtype: int64
       min   max  count
code                   
CHN   2000  2025     26
USA   2000  2025     26
country                   0
code                      0
is_aggregate              0
year                      0
gdp_usd                   4
population                0
inflation_cpi_pct         2
inflation_deflator_pct    4
dtype: int64
     country code  is_aggregate  year       gdp_usd    population  inflation_cpi_pct  inflation_deflator_pct
2680   China  CHN         False  2000  1.211332e+12  1.262645e+09           0.347811                2.062793
2681   China  CHN         False  2001  1.339401e+12  1.271850e+09           0.719126                2.047049
2682   China  CHN         False  2002  1.470558e+12  1.280400e+09          -0.731971                0.602099
2683   China  CHN         False  2003  1.660281e+12  1.288400e+09           1.127603                2.603178
2684   China  CHN         False  2004  1.955347e+12  1.296075e+09      

In [ ]:
# [C4]
print(w2[w2['code']=='USA'].sort_values('year').to_string())
print("\n缺失明细：")
for c_ in ['gdp_usd','inflation_cpi_pct','inflation_deflator_pct','population']:
    for k in ['CHN','USA']:
        miss = w2[(w2['code']==k) & (w2[c_].isna())]['year'].tolist()
        print(c_, k, len(miss), miss)

             country code  is_aggregate  year       gdp_usd   population  inflation_cpi_pct  inflation_deflator_pct
16508  United States  USA         False  2000  1.025095e+13  282162411.0           3.376857                2.265195
16509  United States  USA         False  2001  1.058193e+13  284968955.0           2.826171                2.251690
16510  United States  USA         False  2002  1.092911e+13  287625193.0           1.586032                1.553995
16511  United States  USA         False  2003  1.145645e+13  290107933.0           2.270095                1.974315
16512  United States  USA         False  2004  1.221720e+13  292805298.0           2.677237                2.689088
16513  United States  USA         False  2005  1.303920e+13  295516599.0           3.392747                3.135455
16514  United States  USA         False  2006  1.381558e+13  298379912.0           3.225944                3.083837
16515  United States  USA         False  2007  1.447423e+13  301231207.0

In [ ]:
# [C5]
# 检查重复、零值、完整可比年份
dup = w2.duplicated(subset=['code','year']).sum()
zero_gdp = ((w2['gdp_usd']==0) | (w2['inflation_deflator_pct']==0) | (w2['inflation_cpi_pct']==0)).sum()
print("重复行:", dup, " 零值(可能表示缺失):", zero_gdp)

full = w2.dropna(subset=['gdp_usd','inflation_deflator_pct','population'])
print("\n四字段/三字段齐全（含平减指数）行数：")
print(full.groupby('code')['year'].agg(['min','max','count','nunique']))
cpi_ok = w2.dropna(subset=['inflation_cpi_pct'])
print("\nCPI 通胀非缺失：")
print(cpi_ok.groupby('code')['year'].agg(['min','max','count']))

# 实际 GDP 指数（以 2000=100 的定基，用平减指数累积）可以覆盖的年份
g = w2.dropna(subset=['gdp_usd','inflation_deflator_pct'])
piv = g.pivot(index='year', columns='code', values=['gdp_usd','inflation_deflator_pct'])
print("\n两国同时可用的年份（gdp+deflator）：")
both = piv.dropna()
print(both.index.min(), both.index.max(), len(both))
print(both.index.tolist())

重复行: 0  零值(可能表示缺失): 0

四字段/三字段齐全（含平减指数）行数：
       min   max  count  nunique
code                            
CHN   2000  2023     24       24
USA   2000  2023     24       24

CPI 通胀非缺失：
       min   max  count
code                   
CHN   2000  2024     25
USA   2000  2024     25

两国同时可用的年份（gdp+deflator）：
2000 2023 24
[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


In [ ]:
# [C6]
tab = pd.DataFrame({
    'field':['gdp_usd','population','inflation_cpi_pct','inflation_deflator_pct'],
    'CHN_avail':[24,26,25,24],
    'USA_avail':[24,26,25,24]
})
tab['共缺年份'] = ['2024-2025','无','2025','2024-2025']
print(tab.to_string(index=False))
print("\n可比完整样本：2000-2023 共 24 个年份，增长观测 23 个（需上年基期）")
print("样本期终点（用于实际GDP增速）：2023 年")
print("中国 2000 名义GDP = %.4f 万亿美元, 美国 = %.4f 万亿美元" % (both.loc[2000,('gdp_usd','CHN')]/1e12, both.loc[2000,('gdp_usd','USA')]/1e12))
print("中国 2023 名义GDP = %.4f 万亿美元, 美国 = %.4f 万亿美元" % (both.loc[2023,('gdp_usd','CHN')]/1e12, both.loc[2023,('gdp_usd','USA')]/1e12))

                 field  CHN_avail  USA_avail      共缺年份
               gdp_usd         24         24 2024-2025
            population         26         26         无
     inflation_cpi_pct         25         25      2025
inflation_deflator_pct         24         24 2024-2025

可比完整样本：2000-2023 共 24 个年份，增长观测 23 个（需上年基期）
样本期终点（用于实际GDP增速）：2023 年
中国 2000 名义GDP = 1.2113 万亿美元, 美国 = 10.2510 万亿美元
中国 2023 名义GDP = 17.7948 万亿美元, 美国 = 27.3609 万亿美元


In [ ]:
# [C7]
print(res.to_string(index=False))
print("\nworld_macro 原始全表 CHN/USA 是否有多条记录/重复 code:")
print(wm[wm['code'].isin(['CHN','USA'])].groupby('code').agg(rows=('year','size'), agg_true=('is_aggregate','sum'), ymin=('year','min'), ymax=('year','max')))

code     years  n  nom_cagr  real_cagr  realpc_cagr     nom_cum   real_cum  real_mult   defl_cum  nom_minus_real  real_g_std
 CHN 2000-2023 24 12.393299   8.992146     8.467952 1369.026419 624.585513   7.245855 106.922352        3.401153    5.997249
 USA 2000-2023 24  4.360868   2.029600     1.247988  166.911161  58.745787   1.587458  71.946119        2.331269    1.766320
 JPN 2000-2023 24 -0.714512  -0.515258    -0.435160  -15.204495 -11.202866   0.887971  -5.782526       -0.199253    8.563042
 DEU 2000-2023 24  3.663222   1.748565     1.691069  128.753707  48.987820   1.489878  53.010644        1.914656    7.947643

world_macro 原始全表 CHN/USA 是否有多条记录/重复 code:
      rows  agg_true  ymin  ymax
code                            
CHN     66         0  1960  2025
USA     66         0  1960  2025


In [ ]:
# [C8]
for yy in (2000, 2023):
    for k in ('CHN','USA'):
        v = both.loc[yy,('gdp_usd',k)]/1e12
        print(f"{yy} {k} gdp_usd = {v:.4f} trillion USD")

2000 CHN gdp_usd = 1.2113 trillion USD
2000 USA gdp_usd = 10.2510 trillion USD
2023 CHN gdp_usd = 17.7948 trillion USD
2023 USA gdp_usd = 27.3609 trillion USD


In [ ]:
# [C9]
print(both.head(3)); print(both.tail(3)); print(both.isna().sum())
print(both.index.tolist())

           gdp_usd               inflation_deflator_pct          
code           CHN           USA                    CHN       USA
year                                                             
2000  1.211332e+12  1.025095e+13               2.062793  2.265195
2001  1.339401e+12  1.058193e+13               2.047049  2.251690
2002  1.470558e+12  1.092911e+13               0.602099  1.553995
           gdp_usd               inflation_deflator_pct          
code           CHN           USA                    CHN       USA
year                                                             
2021  1.782046e+13  2.359403e+13               4.552327  4.565986
2022  1.788178e+13  2.574411e+13               1.823677  7.129481
2023  1.779478e+13  2.736094e+13              -0.582394  3.598391
                        code
gdp_usd                 CHN     0
                        USA     0
inflation_deflator_pct  CHN     0
                        USA     0
dtype: int64
[2000, 2001, 2002, 2003, 2004,

In [ ]:
# [C10]
nom = both['gdp_usd']          # 现价美元
defg = both['inflation_deflator_pct']  # 平减指数同比 %
yrs_idx = both.index
# 平减指数：2000=1，逐年累乘
P = pd.DataFrame(index=yrs_idx, columns=['CHN','USA'], dtype=float)
for cc in ['CHN','USA']:
    P.loc[2000, cc] = 1.0
    for y in yrs_idx[1:]:
        P.loc[y, cc] = P.loc[y-1, cc] * (1 + defg.loc[y, cc]/100)
real = nom / P               # 2000 年不变价（以美元计价的平减后）
print(P.iloc[[0,1,10,23]].round(4))
print(real.iloc[[0,23]].round(4))

         CHN     USA
year                
2000  1.0000  1.0000
2001  1.0205  1.0225
2010  1.5070  1.2325
2023  2.0274  1.6814
code           CHN           USA
year                            
2000  1.211332e+12  1.025095e+13
2023  8.777134e+12  1.627295e+13


In [ ]:
# [C11]
nom_g = nom.pct_change()*100
real_g = real.pct_change()*100
def cagr_srs(s, n):
    return ((s.iloc[-1]/s.iloc[0])**(1/n) - 1)*100
out=[]
for cc in ['CHN','USA']:
    out.append(dict(code=cc, n_years=len(yrs_idx), n_growth=yrs_idx.size-1,
        real_cagr=cagr_srs(real[cc], yrs_idx.size-1),
        nom_cagr=cagr_srs(nom[cc], yrs_idx.size-1),
        real_g_mean=real_g[cc].mean(), real_g_std=real_g[cc].std(ddof=1),
        nom_g_mean=nom_g[cc].mean(), nom_g_std=nom_g[cc].std(ddof=1),
        real_cum=(real[cc].iloc[-1]/real[cc].iloc[0]-1)*100,
        nom_cum=(nom[cc].iloc[-1]/nom[cc].iloc[0]-1)*100))
tab2=pd.DataFrame(out).set_index('code')
print(tab2.round(2).to_string())
print('\n2023 实际增速与名义增速:')
print(pd.DataFrame({'real_g':real_g.loc[2023],'nom_g':nom_g.loc[2023]}).round(2))
print('\n平减指数累计涨幅 2000->2023 (%):'); print(((P.loc[2023]-1)*100).round(2))

      n_years  n_growth  real_cagr  nom_cagr  real_g_mean  real_g_std  nom_g_mean  nom_g_std  real_cum  nom_cum
code                                                                                                           
CHN        24        23       8.99     12.39         9.15        6.00       12.70        8.6    624.59  1369.03
USA        24        23       2.03      4.36         2.04        1.77        4.39        2.7     58.75   166.91

2023 实际增速与名义增速:
      real_g  nom_g
code               
CHN     0.10  -0.49
USA     2.59   6.28

平减指数累计涨幅 2000->2023 (%):
CHN    102.74
USA     68.14
Name: 2023, dtype: float64


In [ ]:
# [C12]
dh=pd.DataFrame({'real_cagr_pct':[8.99,2.03],'gap_pp':[8.99-2.03,0]},index=['CHN','USA'])
print('实际CAGR差 %.2f pp' % (tab2.loc['CHN','real_cagr']-tab2.loc['USA','real_cagr']))
print('名义CAGR差 %.2f pp' % (tab2.loc['CHN','nom_cagr']-tab2.loc['USA','nom_cagr']))
print('实际增速标准差比 %.2f 倍' % (tab2.loc['CHN','real_g_std']/tab2.loc['USA','real_g_std']))
print('实际GDP 倍数 CHN %.2f / USA %.2f' % (real['CHN'].iloc[-1]/real['CHN'].iloc[0], real['USA'].iloc[-1]/real['USA'].iloc[0]))
import plotly.graph_objects as go
fig=go.Figure()
for cc in ['CHN','USA']:
    fig.add_trace(go.Scatter(x=yrs_idx, y=real[cc]/real[cc].iloc[0]*100, name=cc, mode='lines'))
fig.update_layout(title='不变价（2000年美元）GDP 指数，2000=100', xaxis_title='年份', yaxis_title='指数')
fig.show()

实际CAGR差 6.96 pp
名义CAGR差 8.03 pp
实际增速标准差比 3.40 倍
实际GDP 倍数 CHN 7.25 / USA 1.59


[图表] 不变价（2000年美元）GDP 指数，2000=100

In [ ]:
# [C13]
print(pd.DataFrame({'defl_2000':P.iloc[0],'defl_2023':P.iloc[-1],
                    'real_2000':real.iloc[0],'real_2023':real.iloc[-1]}))
print('2023 deflator pct:', defg.loc[2023].round(2).to_dict())

     defl_2000  defl_2023     real_2000     real_2023
CHN        1.0   2.027402  1.211332e+12  8.777134e+12
USA        1.0   1.681375  1.025095e+13  1.627295e+13
2023 deflator pct: {'CHN': -0.58, 'USA': 3.6}


In [ ]:
# [C14]
print(real.head(3))
print(real.tail(3))
print('realpc head/tail:'); print(realpc.head(3)); print(realpc.tail(3))
print(tab2.T)
print(w2[['code','year','population','gdp_usd']].head(3))

code           CHN           USA
year                            
2000  1.211332e+12  1.025095e+13
2001  1.312533e+12  1.034890e+13
2002  1.432434e+12  1.052488e+13
code           CHN           USA
year                            
2021  8.897972e+12  1.557398e+13
2022  8.768679e+12  1.586231e+13
2023  8.777134e+12  1.627295e+13
realpc head/tail:
year
2000    23776.507148
2001    23374.885034
2002    24558.782682
dtype: float64
year
2021    37847.788403
2022    34047.468412
2023    34966.550761
dtype: float64
code                 CHN         USA
n_years        24.000000   24.000000
n_growth       23.000000   23.000000
real_cagr       8.992146    2.029600
nom_cagr       12.393299    4.360868
real_g_mean     9.150398    2.044398
real_g_std      5.997249    1.766320
nom_g_mean     12.704182    4.394287
nom_g_std       8.601975    2.696906
real_cum      624.585513   58.745787
nom_cum      1369.026419  166.911161
     code  year    population       gdp_usd
2680  CHN  2000  1.262645e+09  1.21

In [ ]:
# [C15]
print(realpc.name, list(realpc.index[:3]))
print(w2.pivot_table(index='year', columns='code', values='population'))
# 检查 realpc 是由哪个口径算出的
pop = w2.pivot_table(index='year', columns='code', values='population')/1e6
print(pop.head(3)); print(pop.tail(2))
chk = real/pop/1e6
print(chk.head(2)); print(chk.tail(2))

None [2000, 2001, 2002]
code           CHN          USA
year                           
2000  1.262645e+09  282162411.0
2001  1.271850e+09  284968955.0
2002  1.280400e+09  287625193.0
2003  1.288400e+09  290107933.0
2004  1.296075e+09  292805298.0
2005  1.303720e+09  295516599.0
2006  1.311020e+09  298379912.0
2007  1.317885e+09  301231207.0
2008  1.324655e+09  304093966.0
2009  1.331260e+09  306771529.0
2010  1.337705e+09  309378227.0
2011  1.345035e+09  311839461.0
2012  1.354190e+09  314339099.0
2013  1.363240e+09  316726282.0
2014  1.371860e+09  319257560.0
2015  1.379860e+09  321815121.0
2016  1.387790e+09  324353340.0
2017  1.396215e+09  326608609.0
2018  1.402760e+09  328529577.0
2019  1.407745e+09  330226227.0
2020  1.411100e+09  331578104.0
2021  1.412360e+09  332100166.0
2022  1.412175e+09  333996304.0
2023  1.410710e+09  336755052.0
2024  1.408975e+09  340003797.0
2025  1.406585e+09  341784857.0
code       CHN         USA
year                      
2000  1262.645  282.162411

In [ ]:
# [C16]
pop = w2.pivot_table(index='year', columns='code', values='population')
pop = pop.loc[real.index]
realpc2 = real/pop            # 2000年不变价美元 / 人
pc_g = realpc2.pct_change()*100
tot_g = real.pct_change()*100
pop_g = pop.pct_change()*100

def cagrp(s): return ((s.iloc[-1]/s.iloc[0])**(1/(len(s)-1))-1)*100

res3 = pd.DataFrame({
 'pc_g_cagr': [cagrp(realpc2[c]) for c in ['CHN','USA']],
 'pc_cum_pct': [(realpc2[c].iloc[-1]/realpc2[c].iloc[0]-1)*100 for c in ['CHN','USA']],
 'pc_ratio': [realpc2[c].iloc[-1]/realpc2[c].iloc[0] for c in ['CHN','USA']],
 'pc_g_mean': [pc_g[c].mean() for c in ['CHN','USA']],
 'pc_g_std': [pc_g[c].std() for c in ['CHN','USA']],
 'tot_g_cagr': [cagrp(real[c]) for c in ['CHN','USA']],
 'pop_g_cagr': [cagrp(pop[c]) for c in ['CHN','USA']],
 'pc_2000': [realpc2[c].iloc[0] for c in ['CHN','USA']],
 'pc_2023': [realpc2[c].iloc[-1] for c in ['CHN','USA']],
}, index=['CHN','USA'])
res3['contrib_pp'] = res3['tot_g_cagr'] - res3['pc_g_cagr']
res3['pop_contrib_share'] = res3['contrib_pp']/res3['tot_g_cagr']*100
print(res3.round(4).T.to_string())
print('样本期', real.index[0], real.index[-1], '年数', len(real), '增速观测', yrs)
print('人均增速差(中国-美国) CAGR pp:', round(res3.loc['CHN','pc_g_cagr']-res3.loc['USA','pc_g_cagr'],3))
print('人均水平差距 2000 美国/中国倍数:', round(res3.loc['USA','pc_2000']/res3.loc['CHN','pc_2000'],2),
      ' 2023:', round(res3.loc['USA','pc_2023']/res3.loc['CHN','pc_2023'],2))
print('人口累计增幅% CHN/USA:', ((pop.iloc[-1]/pop.iloc[0]-1)*100).round(2).to_dict())

                         CHN         USA
pc_g_cagr             8.4680      1.2480
pc_cum_pct          548.5346     33.0109
pc_ratio              6.4853      1.3301
pc_g_mean             8.6209      1.2635
pc_g_std              5.8893      1.8029
tot_g_cagr            8.9921      2.0296
pop_g_cagr            0.4833      0.7720
pc_2000             959.3604  36329.9703
pc_2023            6221.7845  48322.8220
contrib_pp            0.5242      0.7816
pop_contrib_share     5.8295     38.5106
样本期 2000 2023 年数 24 增速观测 23
人均增速差(中国-美国) CAGR pp: 7.22
人均水平差距 2000 美国/中国倍数: 37.87  2023: 7.77
人口累计增幅% CHN/USA: {'CHN': 11.73, 'USA': 19.35}


In [ ]:
# [C17]
print(pc_g.round(2).to_string())
print('\n人均水平(2000不变价美元):')
print(realpc2.round(0).to_string())

code    CHN   USA
year             
2000    NaN   NaN
2001   7.57 -0.04
2002   8.41  0.76
2003   9.35  1.92
2004   9.46  2.89
2005  11.86  2.53
2006  15.20  1.80
2007  19.10  1.04
2008  19.43 -0.83
2009  10.72 -3.43
2010  11.10  1.83
2011  14.16  0.76
2012   9.67  1.48
2013   9.06  1.35
2014   7.66  1.71
2015   4.98  2.13
2016  -0.43  1.02
2017   4.50  1.75
2018   8.55  2.36
2019   1.11  1.97
2020   2.11 -2.62
2021  15.94  5.65
2022  -1.44  1.27
2023   0.20  1.75

人均水平(2000不变价美元):
code     CHN      USA
year                 
2000   959.0  36330.0
2001  1032.0  36316.0
2002  1119.0  36592.0
2003  1223.0  37293.0
2004  1339.0  38372.0
2005  1498.0  39344.0
2006  1726.0  40051.0
2007  2055.0  40467.0
2008  2455.0  40132.0
2009  2718.0  38757.0
2010  3019.0  39466.0
2011  3447.0  39767.0
2012  3780.0  40354.0
2013  4123.0  40898.0
2014  4439.0  41598.0
2015  4660.0  42482.0
2016  4640.0  42917.0
2017  4849.0  43668.0
2018  5263.0  44701.0
2019  5322.0  45581.0
2020  5434.0  44386.0
2021  63

In [ ]:
# [C18]
fig = px.line(realpc2, x=realpc2.index, y=['CHN','USA'],
              labels={'value':'2000年不变价美元/人','year':'年份','variable':'国家'},
              title='中美人均实际GDP（2000年不变价美元，2000-2023）')
fig.update_layout(yaxis_type='log', xaxis_title='年份', yaxis_title='2000年不变价美元/人（对数轴）')
fig.show()
fig2 = go.Figure()
for c,col in [('CHN','#d62728'),('USA','#1f77b4')]:
    fig2.add_bar(x=pc_g.index, y=pc_g[c], name=c)
fig2.update_layout(title='中美人均实际GDP逐年增速（%，2000年不变价美元口径）',
                   xaxis_title='年份', yaxis_title='%', barmode='group')
fig2.show()

[图表] 中美人均实际GDP（2000年不变价美元，2000-2023）

[图表] 中美人均实际GDP逐年增速（%，2000年不变价美元口径）

In [ ]:
# [C19]
import pandas as pd, numpy as np
inf = w2[['country','code','year','inflation_cpi_pct','inflation_deflator_pct']].copy()
print(inf.pivot_table(index='year', columns='code', values=['inflation_cpi_pct','inflation_deflator_pct']).round(2).to_string())

     inflation_cpi_pct       inflation_deflator_pct      
code               CHN   USA                    CHN   USA
year                                                     
2000              0.35  3.38                   2.06  2.27
2001              0.72  2.83                   2.05  2.25
2002             -0.73  1.59                   0.60  1.55
2003              1.13  2.27                   2.60  1.97
2004              3.82  2.68                   6.95  2.69
2005              1.78  3.39                   3.90  3.14
2006              1.65  3.23                   3.93  3.08
2007              4.82  2.85                   7.75  2.71
2008              5.93  3.84                   7.80  1.93
2009             -0.73 -0.36                  -0.21  0.62
2010              3.18  1.64                   6.88  1.22
2011              5.55  3.16                   8.08  2.06
2012              2.62  2.07                   2.33  1.86
2013              2.62  1.46                   2.16  1.70
2014          

In [ ]:
# [C20]
def stats(s, label):
    s = s.dropna()
    imax = s.idxmax(); imin = s.idxmin()
    return dict(口径=label, n=len(s), 起=s.index.min(), 止=s.index.max(),
                均值=round(s.mean(),2), 中位数=round(s.median(),2), 标准差=round(s.std(ddof=1),2),
                最高年=imax, 最高=round(s[imax],2), 最低年=imin, 最低=round(s[imin],2))

res4 = []
for cpi_lab, def_lab in [('CHN','中国'),('USA','美国')]:
    res4.append(stats(inf[inf.code==cpi_lab].set_index('year')['inflation_cpi_pct'], f'{def_lab}-CPI'))
    res4.append(stats(inf[inf.code==cpi_lab].set_index('year')['inflation_deflator_pct'], f'{def_lab}-平减指数'))
res4 = pd.DataFrame(res4)
print(res4.to_string(index=False))

     口径  n    起    止   均值  中位数  标准差  最高年   最高  最低年    最低
 中国-CPI 25 2000 2024 2.02 1.92 1.71 2008 5.93 2002 -0.73
中国-平减指数 24 2000 2023 3.11 2.25 2.68 2011 8.08 2023 -0.58
 美国-CPI 25 2000 2024 2.58 2.44 1.63 2022 8.00 2009 -0.36
美国-平减指数 24 2000 2023 2.29 1.95 1.36 2022 7.13 2009  0.62


In [ ]:
# [C21]
res4b = []
for code,lab in [('CHN','中国'),('USA','美国')]:
    d = inf[inf.code==code].set_index('year').loc[2000:2023]
    res4b.append(stats(d['inflation_cpi_pct'], f'{lab}-CPI'))
    res4b.append(stats(d['inflation_deflator_pct'], f'{lab}-平减指数'))
res4b = pd.DataFrame(res4b)
print("=== 统一 2000-2023 样本 ===")
print(res4b.to_string(index=False))
# 波动性比值与差距
d = res4b.set_index('口径')
print("\nCPI 均值差(美国-中国) pp:", round(d.loc['美国-CPI','均值']-d.loc['中国-CPI','均值'],2))
print("平减均值差(美国-中国) pp:", round(d.loc['美国-平减指数','均值']-d.loc['中国-平减指数','均值'],2))
print("CPI 标准差比 中国/美国:", round(d.loc['中国-CPI','标准差']/d.loc['美国-CPI','标准差'],2),
      " 平减标准差比:", round(d.loc['中国-平减指数','标准差']/d.loc['美国-平减指数','标准差'],2))
# 2021-2022 美国通胀峰值贡献
for code,lab in [('CHN','中国'),('USA','美国')]:
    d2 = inf[inf.code==code].set_index('year').loc[2000:2023,'inflation_cpi_pct']
    print(lab, "剔除2021-2022后CPI均值:", round(d2.drop([2021,2022]).mean(),2), "标准差:", round(d2.drop([2021,2022]).std(ddof=1),2))

=== 统一 2000-2023 样本 ===
     口径  n    起    止   均值  中位数  标准差  最高年   最高  最低年    最低
 中国-CPI 24 2000 2023 2.09 1.95 1.71 2008 5.93 2002 -0.73
中国-平减指数 24 2000 2023 3.11 2.25 2.68 2011 8.08 2023 -0.58
 美国-CPI 24 2000 2023 2.56 2.36 1.66 2022 8.00 2009 -0.36
美国-平减指数 24 2000 2023 2.29 1.95 1.36 2022 7.13 2009  0.62

CPI 均值差(美国-中国) pp: 0.47
平减均值差(美国-中国) pp: -0.82
CPI 标准差比 中国/美国: 1.03  平减标准差比: 1.97
中国 剔除2021-2022后CPI均值: 2.15 标准差: 1.77
美国 剔除2021-2022后CPI均值: 2.22 标准差: 1.13


In [ ]:
# [C22]
res4b = res4b.set_index('口径')
gap_cpi = res4b.loc['美国-CPI','均值'] - res4b.loc['中国-CPI','均值']
gap_def = abs(res4b.loc['美国-平减指数','均值'] - res4b.loc['中国-平减指数','均值'])
def_ratio = res4b.loc['中国-平减指数','标准差']/res4b.loc['美国-平减指数','标准差']
print(f"平减指数均值: 中国 {res4b.loc['中国-平减指数','均值']:.2f}%, 美国 {res4b.loc['美国-平减指数','均值']:.2f}%, 中国高 {gap_def:.2f}pp")
print(f"平减指数标准差: 中国 {res4b.loc['中国-平减指数','标准差']:.2f}pp, 美国 {res4b.loc['美国-平减指数','标准差']:.2f}pp, 中国/美国={def_ratio:.2f}倍")
print(f"CPI标准差: 中国 {res4b.loc['中国-CPI','标准差']:.2f}pp, 美国 {res4b.loc['美国-CPI','标准差']:.2f}pp, 比值 {res4b.loc['中国-CPI','标准差']/res4b.loc['美国-CPI','标准差']:.2f}")
print()
print("=== 极值年份明细 ===")
print(res4b[['最高年','最高','最低年','最低','中位数']].to_string())

平减指数均值: 中国 3.11%, 美国 2.29%, 中国高 0.82pp
平减指数标准差: 中国 2.68pp, 美国 1.36pp, 中国/美国=1.97倍
CPI标准差: 中国 1.71pp, 美国 1.66pp, 比值 1.03

=== 极值年份明细 ===
          最高年    最高   最低年    最低   中位数
口径                                   
中国-CPI   2008  5.93  2002 -0.73  1.95
中国-平减指数  2011  8.08  2023 -0.58  2.25
美国-CPI   2022  8.00  2009 -0.36  2.36
美国-平减指数  2022  7.13  2009  0.62  1.95


In [ ]:
# [C23]
print("=== 平减指数极值 ===")
for code,lab in [('CHN','中国'),('USA','美国')]:
    s = inf[inf.code==code].set_index('year')['inflation_deflator_pct'].loc[2000:2023].dropna()
    print(f"{lab}: 最高 {s.idxmax()} {s.max():.2f}%, 最低 {s.idxmin()} {s.min():.2f}%, 中位数 {s.median():.2f}%, 均值 {s.mean():.2f}%")

print("\n=== CPI 2024（gdp_usd/平减指数该年缺失，仅CPI可用） ===")
r24 = inf[inf.year==2024][['code','inflation_cpi_pct']]
print(r24.to_string(index=False))

print("\n=== 剔除 2021-2022 后 CPI（美国波动主要来自疫情后） ===")
for code,lab in [('CHN','中国'),('USA','美国')]:
    s = inf[inf.code==code].set_index('year')['inflation_cpi_pct'].loc[2000:2023].dropna()
    s2 = s.drop(index=[2021,2022])
    print(f"{lab}: 全样本 均值{s.mean():.2f}% 标准差{s.std():.2f}pp | 剔除2021-22 均值{s2.mean():.2f}% 标准差{s2.std():.2f}pp")

=== 平减指数极值 ===
中国: 最高 2011 8.08%, 最低 2023 -0.58%, 中位数 2.25%, 均值 3.11%
美国: 最高 2022 7.13%, 最低 2009 0.62%, 中位数 1.95%, 均值 2.29%

=== CPI 2024（gdp_usd/平减指数该年缺失，仅CPI可用） ===
code  inflation_cpi_pct
 CHN           0.218129
 USA           2.949525

=== 剔除 2021-2022 后 CPI（美国波动主要来自疫情后） ===
中国: 全样本 均值2.09% 标准差1.71pp | 剔除2021-22 均值2.15% 标准差1.77pp
美国: 全样本 均值2.56% 标准差1.66pp | 剔除2021-22 均值2.22% 标准差1.13pp


In [ ]:
# [C24]
cpi = inf.pivot_table(index='year', columns='code', values='inflation_cpi_pct').loc[2000:2023]
full = cpi.dropna()
sub = cpi.drop(index=[2021,2022]).dropna()
print("全样本 2000-2023：中国均值 %.2f%% 美国均值 %.2f%%，美国-中国=%.2fpp；标准差 中国 %.2fpp 美国 %.2fpp"%(
    full.CHN.mean(), full.USA.mean(), full.USA.mean()-full.CHN.mean(), full.CHN.std(), full.USA.std()))
print("剔除 2021-2022：中国均值 %.2f%% 美国均值 %.2f%%，美国-中国=%.2fpp；标准差 中国 %.2fpp 美国 %.2fpp"%(
    sub.CHN.mean(), sub.USA.mean(), sub.USA.mean()-sub.CHN.mean(), sub.CHN.std(), sub.USA.std()))
print("均值差变化：%.2fpp -> %.2fpp"%(full.USA.mean()-full.CHN.mean(), sub.USA.mean()-sub.CHN.mean()))

全样本 2000-2023：中国均值 2.09% 美国均值 2.56%，美国-中国=0.47pp；标准差 中国 1.71pp 美国 1.66pp
剔除 2021-2022：中国均值 2.15% 美国均值 2.22%，美国-中国=0.07pp；标准差 中国 1.77pp 美国 1.13pp
均值差变化：0.47pp -> 0.07pp


In [ ]:
# [C25]
import pandas as pd, numpy as np
names = ['real','real_g','nom_g','P','tab2','inf','cpi','cpi_g','defg','realpc2','pop','yrs_idx','w2']
for n in names:
    v = globals().get(n, None)
    print(n, type(v), getattr(v,'shape',None))
print('---- real_g head/tail ----')
print(real_g.head(3)); print(real_g.tail(3))
print('---- cpi ----')
print(cpi.head(3)); print(cpi.tail(3))
print('---- inf head ----')
print(inf.head(5))

real <class 'pandas.DataFrame'> (24, 2)
real_g <class 'pandas.DataFrame'> (24, 2)
nom_g <class 'pandas.DataFrame'> (24, 2)
P <class 'pandas.DataFrame'> (24, 2)
tab2 <class 'pandas.DataFrame'> (2, 10)
inf <class 'pandas.DataFrame'> (52, 5)
cpi <class 'pandas.DataFrame'> (24, 2)
cpi_g <class 'NoneType'> None
defg <class 'pandas.DataFrame'> (24, 2)
realpc2 <class 'pandas.DataFrame'> (24, 2)
pop <class 'pandas.DataFrame'> (24, 2)
yrs_idx <class 'pandas.Index'> (24,)
w2 <class 'pandas.DataFrame'> (52, 8)
---- real_g head/tail ----
code       CHN       USA
year                    
2000       NaN       NaN
2001  8.354529  0.955538
2002  9.135094  1.700447
code        CHN       USA
year                     
2021  16.045979  5.819187
2022  -1.453057  1.851330
2023   0.096419  2.588829
---- cpi ----
code       CHN       USA
year                    
2000  0.347811  3.376857
2001  0.719126  2.826171
2002 -0.731971  1.586032
code       CHN       USA
year                    
2021  0.981015  4.697859

In [ ]:
# [C26]
segs = [('2000-2007',2000,2007),('2008-2019',2008,2019),('2020-2023',2020,2023)]
rows=[]
for lab,a,b in segs:
    for c in ['CHN','USA']:
        rg = real_g.loc[(real_g.index>=a)&(real_g.index<=b), c]
        cp = cpi.loc[(cpi.index>=a)&(cpi.index<=b), c]
        df = defg.loc[(defg.index>=a)&(defg.index<=b), c]
        rows.append(dict(seg=lab, code=c, n_g=rg.notna().sum(),
                         real_g_mean=rg.mean(), cpi_mean=cp.mean(), def_mean=df.mean(),
                         cpi_std=cp.std(), def_std=df.std()))
seg_tab = pd.DataFrame(rows)
pd.set_option('display.width',200)
print(seg_tab.round(2).to_string(index=False))

# 差值（中国-美国）与分化
piv = seg_tab.pivot(index='seg', columns='code')
out5 = pd.DataFrame({
    'real_g_CHN':piv[('real_g_mean','CHN')], 'real_g_USA':piv[('real_g_mean','USA')],
    'real_gap_pp':piv[('real_g_mean','CHN')]-piv[('real_g_mean','USA')],
    'cpi_CHN':piv[('cpi_mean','CHN')], 'cpi_USA':piv[('cpi_mean','USA')],
    'cpi_gap_pp':piv[('cpi_mean','CHN')]-piv[('cpi_mean','USA')],
})
print('\n', out5.round(2).to_string())

      seg code  n_g  real_g_mean  cpi_mean  def_mean  cpi_std  def_std
2000-2007  CHN    7        12.25      1.69      3.73     1.82     2.49
2000-2007  USA    7         2.51      2.78      2.46     0.61     0.55
2008-2019  CHN   12         8.97      2.59      3.21     1.78     2.94
2008-2019  USA   12         1.78      1.77      1.56     1.15     0.51
2020-2023  CHN    4         4.26      1.40      1.57     0.98     2.22
2020-2023  USA    4         2.01      4.51      4.16     2.78     2.40

            real_g_CHN  real_g_USA  real_gap_pp  cpi_CHN  cpi_USA  cpi_gap_pp
seg                                                                         
2000-2007       12.25        2.51         9.74     1.69     2.78       -1.08
2008-2019        8.97        1.78         7.19     2.59     1.77        0.82
2020-2023        4.26        2.01         2.25     1.40     4.51       -3.11


In [ ]:
# [C27]
last = inf[inf.year.between(2020,2024)].pivot(index='year', columns='code')[['inflation_cpi_pct','inflation_deflator_pct']]
print(last.round(2).to_string())

cpi20_24 = inf[(inf.year>=2020)&(inf.year<=2024)].groupby('code')['inflation_cpi_pct'].agg(['count','mean','std'])
print('\nCPI 2020-2024 均值:\n', cpi20_24.round(2).to_string())
print('2020-2024 CPI 差(中国-美国):', round(cpi20_24.loc['CHN','mean']-cpi20_24.loc['USA','mean'],2), 'pp')

# 阶段变化幅度
print('\n中国实际增速分段降幅: 2000-07 -> 2008-19',
      round(out5.loc['2008-2019','real_g_CHN']-out5.loc['2000-2007','real_g_CHN'],2),
      'pp; 2008-19 -> 2020-23',
      round(out5.loc['2020-2023','real_g_CHN']-out5.loc['2008-2019','real_g_CHN'],2),'pp')
print('美国实际增速分段变化: 2000-07 -> 2008-19',
      round(out5.loc['2008-2019','real_g_USA']-out5.loc['2000-2007','real_g_USA'],2),
      'pp; 2008-19 -> 2020-23',
      round(out5.loc['2020-2023','real_g_USA']-out5.loc['2008-2019','real_g_USA'],2),'pp')
print('实际增速差各段: ', [round(v,2) for v in out5['real_gap_pp']])
print('CPI 差各段: ', [round(v,2) for v in out5['cpi_gap_pp']])
print('平减指数差各段:', [round(seg_tab.pivot(index='seg',columns='code')['def_mean'].diff(axis=1)['USA'].get(s,np.nan),2) if False else None for s in []])
piv2 = seg_tab.pivot(index='seg', columns='code')['def_mean']
print('平减指数 CHN-USA 差: ', [round(piv2.loc[s,'CHN']-piv2.loc[s,'USA'],2) for s in piv2.index])
print('\n最明显分化: 增速差最大段', out5['real_gap_pp'].idxmax(), round(out5['real_gap_pp'].max(),2),
      '; CPI差最大(绝对值)段', out5['cpi_gap_pp'].abs().idxmax(), round(out5['cpi_gap_pp'].abs().max(),2))

     inflation_cpi_pct       inflation_deflator_pct      
code               CHN   USA                    CHN   USA
year                                                     
2020              2.42  1.23                   0.49  1.33
2021              0.98  4.70                   4.55  4.57
2022              1.97  8.00                   1.82  7.13
2023              0.23  4.12                  -0.58  3.60
2024              0.22  2.95                    NaN   NaN

CPI 2020-2024 均值:
       count  mean  std
code                  
CHN       5  1.17  1.0
USA       5  4.20  2.5
2020-2024 CPI 差(中国-美国): -3.03 pp

中国实际增速分段降幅: 2000-07 -> 2008-19 -3.27 pp; 2008-19 -> 2020-23 -4.71 pp
美国实际增速分段变化: 2000-07 -> 2008-19 -0.73 pp; 2008-19 -> 2020-23 0.23 pp
实际增速差各段:  [9.74, 7.19, 2.25]
CPI 差各段:  [-1.08, 0.82, -3.11]
平减指数差各段: []
平减指数 CHN-USA 差:  [np.float64(1.27), np.float64(1.65), np.float64(-2.58)]

最明显分化: 增速差最大段 2000-2007 9.74 ; CPI差最大(绝对值)段 2020-2023 3.11


In [ ]:
# [C28]
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig5 = make_subplots(rows=1, cols=2, subplot_titles=('实际 GDP 增速均值（%，2000年不变价美元）','CPI 通胀均值（%）'))
seg_lab = ['2000-2007','2008-2019','2020-2023']
for c,colr in [('CHN','#d62728'),('USA','#1f77b4')]:
    nm = '中国' if c=='CHN' else '美国'
    fig5.add_trace(go.Bar(x=seg_lab, y=[out5.loc[s,'real_g_'+c] for s in seg_lab], name=nm,
                          marker_color=colr, text=[f"{out5.loc[s,'real_g_'+c]:.2f}" for s in seg_lab],
                          textposition='top center'), row=1, col=1)
    fig5.add_trace(go.Bar(x=seg_lab, y=[out5.loc[s,'cpi_'+c] for s in seg_lab], name=nm,
                          marker_color=colr, showlegend=False,
                          text=[f"{out5.loc[s,'cpi_'+c]:.2f}" for s in seg_lab],
                          textposition='top center'), row=1, col=2)
fig5.update_yaxes(title_text='%', row=1, col=1); fig5.update_yaxes(title_text='%', row=1, col=2)
fig5.update_layout(title='中美分阶段实际增速与CPI通胀均值对比（2000-2023）', barmode='group', height=450)
fig5.show()

ValueError: 
    Invalid value of type 'builtins.str' received for the 'textposition' property of bar
        Received value: 'top center'

    The 'textposition' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['inside', 'outside', 'auto', 'none']
      - A tuple, list, or one-dimensional numpy array of the above

In [ ]:
# [C29]
fig5 = make_subplots(rows=1, cols=2, subplot_titles=('实际 GDP 增速均值（%，2000年不变价美元）','CPI 通胀均值（%）'))
for c,colr in [('CHN','#d62728'),('USA','#1f77b4')]:
    nm = '中国' if c=='CHN' else '美国'
    fig5.add_trace(go.Bar(x=seg_lab, y=[out5.loc[s,'real_g_'+c] for s in seg_lab], name=nm,
                          marker_color=colr, text=[f"{out5.loc[s,'real_g_'+c]:.2f}" for s in seg_lab],
                          textposition='outside'), row=1, col=1)
    fig5.add_trace(go.Bar(x=seg_lab, y=[out5.loc[s,'cpi_'+c] for s in seg_lab], name=nm,
                          marker_color=colr, showlegend=False,
                          text=[f"{out5.loc[s,'cpi_'+c]:.2f}" for s in seg_lab],
                          textposition='outside'), row=1, col=2)
fig5.update_yaxes(title_text='%', row=1, col=1); fig5.update_yaxes(title_text='%', row=1, col=2)
fig5.update_layout(title='中美分阶段实际增速与CPI通胀均值对比（2000-2023）', barmode='group', height=450)
fig5.show()
print('分段样本量（增速观测数）:', dict(zip(seg_lab,[int(seg_tab[(seg_tab.seg==s)&(seg_tab.code=='CHN')].n_g.iloc[0]) for s in seg_lab])))

分段样本量（增速观测数）: {'2000-2007': 7, '2008-2019': 12, '2020-2023': 4}


[图表] 中美分阶段实际增速与CPI通胀均值对比（2000-2023）

In [ ]:
# [C30]
print(type(real_g)); print(real_g.head()); print(real_g.shape)
print(cpi.head(3), cpi.shape)
print(inf.head(3))

<class 'pandas.DataFrame'>
code        CHN       USA
year                     
2000        NaN       NaN
2001   8.354529  0.955538
2002   9.135094  1.700447
2003  10.036967  2.795606
2004  10.116761  3.847772
(24, 2)
code       CHN       USA
year                    
2000  0.347811  3.376857
2001  0.719126  2.826171
2002 -0.731971  1.586032 (24, 2)
     country code  year  inflation_cpi_pct  inflation_deflator_pct
2680   China  CHN  2000           0.347811                2.062793
2681   China  CHN  2001           0.719126                2.047049
2682   China  CHN  2002          -0.731971                0.602099


In [ ]:
# [C31]
g = real_g.dropna(how="all")  # 2001-2023
rows=[]
for c in ["CHN","USA"]:
    sub = pd.concat([g[c].rename("g"), cpi[c].rename("cpi"), inf.set_index(["code","year"]).loc[(c,), "inflation_deflator_pct"].rename("def")], axis=1).dropna()
    n=len(sub)
    r_p = sub["g"].corr(sub["cpi"])
    r_s = sub["g"].corr(sub["cpi"], method="spearman")
    r_d = sub["g"].corr(sub["def"])
    r_ds = sub["g"].corr(sub["def"], method="spearman")
    rows.append(dict(code=c, n=n, start=sub.index.min(), end=sub.index.max(),
                     r_pearson_cpi=r_p, r_spearman_cpi=r_s, r_pearson_def=r_d, r_spearman_def=r_ds))
cor = pd.DataFrame(rows)
print(cor.round(4).to_string(index=False))
print()
# 同时给出两变量各自的均值标准差，便于解释
for c in ["CHN","USA"]:
    sub = pd.concat([g[c].rename("g"), cpi[c].rename("cpi")], axis=1).dropna()
    print(c, "均值 g=%.2f cpi=%.2f  std g=%.2f cpi=%.2f" % (sub.g.mean(), sub.cpi.mean(), sub.g.std(), sub.cpi.std()))

code  n  start  end  r_pearson_cpi  r_spearman_cpi  r_pearson_def  r_spearman_def
 CHN 23   2001 2023         0.4136          0.2915         0.7027          0.7075
 USA 23   2001 2023         0.3380          0.2717         0.3881          0.3953

CHN 均值 g=9.15 cpi=2.17  std g=6.00 cpi=1.70
USA 均值 g=2.04 cpi=2.53  std g=1.77 cpi=1.69


In [ ]:
# [C32]
from scipy import stats
out=[]
for c in ["CHN","USA"]:
    sub = pd.concat([g[c].rename("g"), cpi[c].rename("cpi")], axis=1).dropna()
    for lab_x, s in [("CPI", sub["cpi"])]:
        r,p = stats.pearsonr(sub["g"], s)
        rho,ps = stats.spearmanr(sub["g"], s)
        out.append(dict(code=c, var=lab_x, n=len(sub), r=r, p_pearson=p, rho=rho, p_spearman=ps))
    d = inf.set_index(["code","year"]).loc[(c,), "inflation_deflator_pct"]
    sub2 = pd.concat([g[c].rename("g"), d.rename("def")], axis=1).dropna()
    r,p = stats.pearsonr(sub2["g"], sub2["def"]); rho,ps = stats.spearmanr(sub2["g"], sub2["def"])
    out.append(dict(code=c, var="deflator", n=len(sub2), r=r, p_pearson=p, rho=rho, p_spearman=ps))
sig = pd.DataFrame(out)
print(sig.round(4).to_string(index=False))

# 稳健性：剔除 2020（疫情）与 2008-2009 危机年
print("\n--- 稳健性: 剔除 2008,2009,2020 ---")
rob=[]
for c in ["CHN","USA"]:
    sub = pd.concat([g[c].rename("g"), cpi[c].rename("cpi")], axis=1).dropna()
    s2 = sub.drop(index=[2008,2009,2020])
    r,p = stats.pearsonr(s2["g"], s2["cpi"])
    rob.append(dict(code=c, n=len(s2), r=r, p=p))
print(pd.DataFrame(rob).round(4).to_string(index=False))

code      var  n      r  p_pearson    rho  p_spearman
 CHN      CPI 23 0.4136     0.0498 0.2915      0.1772
 CHN deflator 23 0.7027     0.0002 0.7075      0.0002
 USA      CPI 23 0.3380     0.1147 0.2717      0.2097
 USA deflator 23 0.3881     0.0672 0.3953      0.0619

--- 稳健性: 剔除 2008,2009,2020 ---
code  n      r      p
 CHN 20 0.3725 0.1058
 USA 20 0.1383 0.5610


In [ ]:
# [C33]
import plotly.graph_objects as go
fig6 = go.Figure()
for c,nmm,clr in [("CHN","中国","#d62728"),("USA","美国","#1f77b4")]:
    sub = pd.concat([g[c].rename("g"), cpi[c].rename("cpi")], axis=1).dropna()
    fig6.add_trace(go.Scatter(x=sub["g"], y=sub["cpi"], mode="markers+text", name=nmm,
                              text=[str(y) for y in sub.index], textposition="top center",
                              marker=dict(color=clr, size=9)))
fig6.update_layout(title="2001-2023 年实际 GDP 增速与 CPI 通胀（散点，点标注年份）<br><sup>实际增速为 2000 年不变价美元口径；相关系数中国 0.41、美国 0.34，仅描述相关</sup>",
                   xaxis_title="实际 GDP 增速（%）", yaxis_title="CPI 通胀率（%）")
fig6.show()
print(sig.assign(r_abs=sig.r.abs()).round(3).to_string(index=False))

code      var  n     r  p_pearson   rho  p_spearman  r_abs
 CHN      CPI 23 0.414      0.050 0.292       0.177  0.414
 CHN deflator 23 0.703      0.000 0.708       0.000  0.703
 USA      CPI 23 0.338      0.115 0.272       0.210  0.338
 USA deflator 23 0.388      0.067 0.395       0.062  0.388


[图表] 2001-2023 年实际 GDP 增速与 CPI 通胀（散点，点标注年份）<br><sup>实际增速为 2000 年不变价美元口径；相关系数中国 0.41、美国 0.34，仅描述相关</sup>

### 已登记结论

- **F1** world_macro 中 CHN、USA（is_aggregate=False）2000-2025 各 26 个年份记录，无重复行、无 is_aggregate 标记错漏；但 gdp_usd 与 inflation_deflator_pct 均在 2024、2025 缺失，两字段各只有 24 个非缺失年份，因此实际 GDP 增速与平减指数的可比完整样本期为 2000-2023（24 年，23 个增长观测）。（证据 C3,C4,C5,C6,C7）
- **F2** 分字段缺失情况（中美完全一致）：gdp_usd 非缺失 24 年（缺 2024、2025）；inflation_deflator_pct 非缺失 24 年（缺 2024、2025）；inflation_cpi_pct 非缺失 25 年（仅缺 2025，可用至 2024 年）；population 非缺失 26 年无缺口。（证据 C4,C6）
- **F3** 现价美元口径下，名义 GDP 中国 2000 年 1.2113、美国 10.2510；2023 年中国 17.7948、美国 27.3609（单位 trillion USD，均为现价名义值）。（证据 C8）
- **F4** 以 inflation_deflator_pct 逐年累乘构造的平减指数（2000=1）折算，2000-2023 年实际（2000 年不变价美元）GDP 年均复合增速：中国 8.99%、美国 2.03%，中国高 6.96 个百分点。样本 24 个年度、23 个增长观测。（证据 C11,C12）
- **F5** 2000-2023 年增速波动率（年度增速标准差）：实际口径中国 6.00 个百分点、美国 1.77 个百分点，中国为美国的 3.40 倍；名义口径中国 8.60、美国 2.70 个百分点。中国实际年均增速（算术均值）9.15%、美国 2.04%。（证据 C11,C12）
- **F6** 同一期间名义（现价美元）GDP 年均复合增速：中国 12.39%、美国 4.36%，中国高 8.03 个百分点。（证据 C11,C12）
- **F7** 2000-2023 年不变价 GDP 累计增至 2000 年的中国 7.25 倍（+624.59%）、美国 1.59 倍（+58.75%），名义累计增幅中国 1369.03%、美国 166.91%；GDP 平减指数累计上涨中国 102.74%、美国 68.14%。（证据 C11,C12）
- **F8** 单年看 2023 年不变价实际 GDP 增速中国仅 0.10%、美国 2.59%，名义（现价美元）增速中国 -0.49%（人民币贬值与低平减指数）、美国 6.28%；中国名义增速低于实际增速，与其 2023 年平减指数为负有关。（证据 C11）
- **F9** 口径提醒：本步"实际 GDP"由现价美元 gdp_usd 除以 2000=1 的平减指数得到，等价于 2000 年不变价美元，因此汇率变动会进入"实际"增速——2023 年中国按此口径实际增速仅 0.10%、平减指数为 -0.58%（美国 3.60%），与该年官方本币不变价增速存在偏离；比较中美长期实际增速时宜同时参考本币不变价口径。（证据 C11,C13）
- **F10** 2000-2023 年（24 年、23 个增速观测、2000 年不变价美元口径）人均实际 GDP 年均复合增速：中国 8.47%、美国 1.25%，中国高 7.22 个百分点。（证据 C16）
- **F11** 2000-2023 年人均实际 GDP 累计涨幅：中国 +548.53%（2023 年为 2000 年的 6.49 倍，从 959 美元升至 6222 美元）、美国 +33.01%（1.33 倍，从 36330 美元升至 48323 美元）；算术均值口径中国 8.62%、美国 1.26%，标准差中国 5.89pp、美国 1.80pp。（证据 C16,C17）
- **F12** 人口增长对总量实际增速的贡献：2000-2023 年人口年均增速中国 0.48%、美国 0.77%，对应贡献总量增速 0.52pp（占 5.83%）与 0.78pp（占 38.51%）；即实际 GDP 年均 8.99%/2.03% 中，中国几乎全部来自人均产出，美国近四成来自人口增长。人口累计增幅中国 +11.73%、美国 +19.35%（2000-2023）。（证据 C16）
- **F13** 人均实际 GDP 差距大幅收敛：美国/中国的人均实际 GDP 倍数由 2000 年的 37.87 倍降至 2023 年的 7.77 倍。（证据 C16,C17）
- **F14** 2000-2023 年（24 年）CPI 通胀：中国均值 2.09%、标准差 1.71pp，美国均值 2.56%、标准差 1.66pp；美国均值高 0.47pp，两国 CPI 波动几乎相同（标准差比 1.03）。（证据 C21）
- **F15** 2000-2023 年 GDP 平减指数通胀：中国均值 3.11%、标准差 2.68pp，美国均值 2.29%、标准差 1.36pp；中国水平高 0.82pp，波动为美国的 1.97 倍。（证据 C22）
- **F16** CPI 通胀极值年份：中国最高 2008 年 5.93%、最低 2002 年 -0.73%（通缩）；美国最高 2022 年 8.00%、最低 2009 年 -0.36%。中位数中国 1.95%、美国 2.36%。（证据 C22）
- **F20** 剔除 2021-2022 疫情后两年，美国 CPI 标准差由 1.66pp 降至 1.13pp、均值由 2.56% 降至 2.22%，中国基本不变（标准差 1.71→1.77pp、均值 2.09→2.15%）。美国 CPI 均值仍略高于中国，但均值差由全样本的 0.47pp 缩小至 0.07pp，说明两国 CPI 均值差与美国 CPI 的额外波动主要来自 2021-2022 的通胀冲击。（证据 C24）
- **F21** 平减指数通胀极值年份：中国最高 2011 年 8.08%、最低 2023 年 -0.58%（GDP 平减指数转负）；美国最高 2022 年 7.13%、最低 2009 年 0.62%。描述性看，美国样本内平减指数低点仍为正、未出现平减指数通缩，中国则出现 2023 年负值。样本 2000-2023 共 24 年。（证据 C23,C21）
- **F22** 分三段的实际GDP增速均值（2000年不变价美元口径）：中国 2000-2007 年 12.25%、2008-2019 年 8.97%、2020-2023 年 4.26%（观测数 7/12/4）；美国对应 2.51%、1.78%、2.01%。中美增速差由 9.74pp 收窄至 7.19pp、再降至 2.25pp。（证据 C26,C27）
- **F23** 分段 CPI 通胀均值：中国 2000-2007 年 1.69%、2008-2019 年 2.59%、2020-2023 年 1.40%；美国 2.78%、1.77%、4.51%。中美 CPI 差（中国-美国）依次为 -1.08pp、+0.82pp、-3.11pp，2020-2023 年符号反转。（证据 C26）
- **F24** 2020-2023 年是两国通胀分化最明显的阶段：美国 CPI 均值 4.51%、标准差 2.78pp，中国 CPI 均值 1.40%、标准差 0.98pp；平减指数口径同期美国 4.16%、中国 1.57%，两国方向与 2008-2019 年（中国平减指数 3.21% 高于美国 1.56%）相反。（证据 C26）
- **F25** 通胀分化集中在 2021-2022 年：美国 CPI 2021 年 4.70%、2022 年 8.00%，中国同期仅 0.98%、1.97%；2020-2024 五年 CPI 均值美国 4.20%、中国 1.17%。同期中国实际增速均值由 2008-2019 年 8.97% 降至 2020-2023 年 4.26%，美国由 1.78% 微升至 2.01%。（证据 C26,C27）
- **F26** 2001-2023 年（23 个年度观测）实际 GDP 增速与 CPI 通胀的 Pearson 相关系数：中国 0.41（p=0.050），美国 0.34（p=0.115）；Spearman 秩相关中国 0.29（p=0.177）、美国 0.27（p=0.210）。即两国增长与 CPI 通胀呈弱正相关，且美国不显著，仅为描述性相关而非因果。（证据 C31,C32）
- **F27** 改用 GDP 平减指数通胀口径，同期（2001-2023，23 个观测）实际增速与通胀的相关系数明显更高：中国 0.70（p=0.0002，Rank 0.71），美国 0.39（p=0.067，Rank 0.40）。中国相关显著、美国边缘不显著，说明中国高频增长与整体平减指数同向变动更明显。（证据 C32）
- **F28** 剔除 2008、2009、2020 三个极端年份后（n=20），实际增速与 CPI 通胀的相关系数下降为：中国 0.37（p=0.106）、美国 0.14（p=0.561），均不再显著。说明原始正相关相当一部分来自危机与疫情年份的共同大幅波动，增长—通胀关联在正常年份并不稳健。（证据 C32）

### 报告

## 结论

2000-2023 年可比样本内，中国经济增长显著快于美国：2000 年不变价美元口径实际 GDP 年均复合增速中国 8.99%、美国 2.03%，且中国增速波动更大 [F4][F5]。通胀结构不同：CPI 均值美国略高于中国（2.56% vs 2.09%），但 GDP 平减指数通胀中国更高（3.11% vs 2.29%）[F14][F15]。2020-2023 年美国 CPI 通胀明显高于中国，同期中国实际增速放缓、中美增速差收窄 [F22][F24]。

## 主要发现

1. **增长差距大，波动也大。** 2000-2023 年（24 年、23 个增长观测）2000 年不变价美元口径实际 GDP 年均复合增速中国 8.99%、美国 2.03%，中国高 6.96 个百分点；名义（现价美元）年均复合增速中国 12.39%、美国 4.36% [F4][F6]。实际增速标准差中国 6.00 个百分点、美国 1.77 个百分点，中国为美国的 3.40 倍 [F5]。不变价 GDP 累计增至 2000 年的中国 7.25 倍（+624.59%）、美国 1.59 倍（+58.75%）[F7]。

[C12]

2. **中国增长几乎全部来自人均产出，美国近四成来自人口。** 人均实际 GDP 年均复合增速中国 8.47%、美国 1.25% [F10]；人口年均增速中国 0.48%、美国 0.77%，按差值法近似的人口贡献分别为 0.52pp 和 0.78pp（占 5.83% 和 38.51%）[F12]。人均实际 GDP 累计涨幅中国 +548.53%、美国 +33.01% [F11]；美国/中国的人均实际 GDP 倍数由 37.87 倍降至 7.77 倍 [F13]。

[C18]

3. **通胀水平与波动：CPI 美国略高，平减指数中国更高。** 2000-2023 年 CPI 均值中国 2.09%、美国 2.56%，标准差中国 1.71pp、美国 1.66pp [F14]；GDP 平减指数通胀均值中国 3.11%、美国 2.29%，标准差中国 2.68pp、美国 1.36pp，中国波动为美国的 1.97 倍 [F15]。CPI 最高中国 2008 年 5.93%、美国 2022 年 8.00%；最低中国 2002 年 -0.73%、美国 2009 年 -0.36% [F16]。平减指数通胀率最高中国 2011 年 8.08%、美国 2022 年 7.13%；最低中国 2023 年 -0.58%、美国 2009 年 0.62% [F21]。

4. **分段看，中美增速差持续收窄，CPI 差呈“负—正—负”。** 实际 GDP 增速三段均值：中国 2000-2007 年 12.25%、2008-2019 年 8.97%、2020-2023 年 4.26%；美国对应 2.51%、1.78%、2.01%；中美差由 9.74pp 收窄至 7.19pp、再降至 2.25pp [F22]。CPI 三段均值中国 1.69%、2.59%、1.40%，美国 2.78%、1.77%、4.51%；中美 CPI 差依次为 -1.08pp、+0.82pp、-3.11pp [F23]。

[C29]

5. **2020-2023 年是通胀分化最明显的阶段。** 美国 CPI 均值 4.51%、中国 1.40%；平减指数口径美国 4.16%、中国 1.57%，中美高低关系由 2008-2019 年中国（3.21%）高于美国（1.56%），转为美国高于中国 [F24]。2021-2022 年美国 CPI 分别为 4.70%、8.00%，中国同期仅 0.98%、1.97%；2020-2024 五年 CPI 均值美国 4.20%、中国 1.17% [F25]。

6. **2023 年单年：中国实际美元增速很低，名义美元增速为负。** 2023 年中国 2000 年不变价美元实际 GDP 增速 0.10%、名义美元增速 -0.49%、平减指数通胀 -0.58%；美国分别为 2.59%、6.28%、3.60% [F8]。该“实际”口径受汇率变动影响，不等同于官方本币不变价增速 [F9]。

7. **增长与通胀仅呈弱正相关，且不稳健。** 2001-2023 年实际增速与 CPI 通胀的 Pearson 相关系数中国 0.41、美国 0.34，美国不显著；Spearman 秩相关中国 0.29、美国 0.27 [F26]。平减指数口径相关系数中国 0.70、美国 0.39 [F27]；但剔除 2008、2009、2020 年后，CPI 口径相关系数降至中国 0.37、美国 0.14，均不显著 [F28]。

[C33]

8. **剔除 2021-2022 后，美国 CPI 波动明显下降。** 美国 CPI 标准差由 1.66pp 降至 1.13pp、均值由 2.56% 降至 2.22%；中国标准差由 1.71pp 升至 1.77pp、均值由 2.09% 升至 2.15% [F20]。

## 方法与口径

样本为 world_macro 中 CHN、USA 的非汇总记录。gdp_usd 与 inflation_deflator_pct 均缺 2024、2025 年，因此实际 GDP 增速与平减指数的可比完整样本期为 2000-2023 年（24 年、23 个增长观测）[F1][F2]。实际 GDP 用现价美元 GDP 除以以 2000=1 逐年累乘构造的平减指数，等价于 2000 年不变价美元，汇率变动会进入“实际”增速 [F4][F9]。人均指标用不变价 GDP 除以 population [F10]。CPI 通胀可延伸至 2024 年 [F2]。相关系数均为描述性相关，不作因果解释 [F26]。

## 假设与局限

- **数据缺失统计的可复现性。** 字段缺失年份来自已登记结论，但其汇总表为手工构造、不可复现，数据更新后可用年份可能变化；本报告因此以 2000-2023 为主样本 [F1][F2]。
- **“实际 GDP”口径。** 该指标为 2000 年不变价美元口径，受汇率变动影响，不等同于官方本币不变价增速；比较中美长期实际增速时宜同时参考本币不变价口径 [F4][F9]。
- **2023 年名义美元负增长不能归因于汇率。** 已登记证据未提供 2023 年汇率数据，只能确认名义美元增速、实际美元增速与平减指数通胀的组合关系，不能把名义美元负增长直接归因于人民币贬值 [F8]。
- **人均差距“收敛”仅指相对倍数。** 美国/中国倍数由 37.87 倍降至 7.77 倍 [F13]，但倍数收窄不等于绝对差额缩小；绝对差额是否缩小未在已登记结论中量化。人均水平与倍数基于市场汇率、非 PPP，不宜直接解读为生活水平差距 [F10][F13]。
- **人口贡献为近似分解。** 0.52pp、0.78pp 是总量与人均增速的差值法近似，含交互项，与人口增速 0.48%、0.77% 不是同一个量 [F12]。
- **CPI 剔除年份的归因是描述性的。** 全样本美国 CPI 标准差 1.66pp 已略低于中国 1.71pp，剔除 2021-2022 后美国 1.13pp 低于中国 1.77pp [F20]；剔除后的变化只是描述性归因，不是因果。
- **平减指数措辞与分布。** 转负的是平减指数通胀率（同比），不是价格指数水平；美国平减指数通胀率样本内未出现负值 [F21]。已登记结论未报告平减指数中位数，分布特征不完整 [F15][F21]。
- **分段比较的限制。** 2020-2023 段仅 4 个观测，且数据止于 2023 年，无法完整覆盖 2020-2025；三段长度 7/12/4 不均衡，4 年段含疫情冲击和低基数，均值对年份构成敏感 [F22]。中美 CPI 差为“负—正—负”，不是简单由正转负；2020-2023 负向最大 [F23]。2020-2023 是描述统计中差距最大的阶段，未做统计检验；平减指数均为正，变化的是中美高低排序而非通胀方向 [F24]。
- **2024 年 CPI 窗口不一致。** CPI 的 2020-2024 五年均值与实际增速的 2020-2023 四年窗口不一致，不能称严格同期；2024 年 CPI 可能为初步值，需核实来源 [F25]。
- **相关分析的稳健性。** 中国 CPI 口径 Pearson 相关仅边缘显著、Spearman 不显著；样本量小，不显著不等于无关联，剔除年份为事后选择，“正常年份”表述过宽 [F26][F28]。平减指数通胀与实际增速存在核算恒等关系（名义≈实际+平减指数），相关含机械成分；年度数据不是高频；平减指数口径未做剔除极端年份的稳健性检验，F28 仅适用于 CPI 口径 [F27][F28]。CPI 数据来源及 2023 年缺失/0 值处理未在已登记结论中说明，若占位值进入年度均值会影响相关结果 [F26]。